# N0: External Dataset Evaluation

Test Model_4_4 on external datasets:
1. **huynhtuan0106**: Train + Test combined (1124 + unknown samples)
2. **vnfd**: Two files combined (223 + 226 samples)

Pipeline:
- Load original training data (to fit TF-IDF & SVD)
- Load external datasets
- Apply strict_clean for TF-IDF, loose_clean for PhoBERT
- Extract embeddings using pre-trained TF-IDF + PhoBERT
- Load best model + scaler
- Predict with threshold 0.45
- Evaluate generalization on external data

## 1. Import Libraries & Setup

In [4]:
# Install required packages FIRST (before any imports)
import subprocess
import sys

print("Installing required packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyvi", "emoji"])
print("✅ Packages installed successfully\n")

# Now import all libraries
import numpy as np
import pandas as pd
import warnings
import torch
import gc
import joblib
import os
import re
from tqdm import tqdm
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModel

# NEW: Import pyvi and emoji for preprocessing
from pyvi import ViTokenizer
import emoji

# Setup device for PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("✅ All imports successful")
print(f"   PyTorch Device: {device}")
print(f"   Mode: EXTERNAL DATASET EVALUATION")
print(f"   NEW: pyvi + emoji libraries loaded for advanced preprocessing")

Installing required packages...
✅ Packages installed successfully

✅ All imports successful
   PyTorch Device: cuda
   Mode: EXTERNAL DATASET EVALUATION
   NEW: pyvi + emoji libraries loaded for advanced preprocessing


## 2. Define Text Cleaning Functions

In [5]:
def strict_clean(text):
    """
    Strict cleaning for TF-IDF (aggressive preprocessing)
    - Decode emojis to text using emoji library
    - Word segmentation + join compound words with _ using pyvi
    - Lowercase
    - Remove URLs, emails
    - Remove hashtag symbols (but keep text)
    - Remove special characters (keep Vietnamese diacritics)
    - REMOVE NUMBERS
    - Remove extra spaces
    """
    if pd.isna(text) or text == '':
        return ''
    
    # 1. Decode emojis to text (😀 → face with smiling eyes)
    text = emoji.replace_emoji(text, replace="")  # Remove emojis entirely for now
    # Alternatively: emoji.demojize(text) converts to :emoji_name: format
    
    # 2. Vietnamese word segmentation + join with underscore
    # This joins multi-word entities: "Việt Nam" → "Việt_Nam"
    text = ViTokenizer.tokenize(text)
    
    # 3. Lowercase
    text = text.lower()
    
    # 4. Remove URLs
    text = re.sub(r'http\S+|www\S+', '[URL]', text)
    text = re.sub(r'<\s*url\s*>', '[URL]', text)
    
    # 5. Remove emails
    text = re.sub(r'\S+@\S+', '[EMAIL]', text)
    
    # 6. Remove hashtag symbols but keep text (#word → word)
    text = re.sub(r'#(\w+)', r'\1', text)
    
    # 7. Remove special characters (keep Vietnamese diacritics and UNDERSCORES from pyvi)
    # \w includes letters, digits, and underscores (which pyvi uses to join words)
    text = re.sub(r'[^\w\s\u0100-\u017f\u0180-\u024f]', '', text, flags=re.UNICODE)
    
    # 8. REMOVE NUMBERS
    text = re.sub(r'\d+', '', text)
    
    # 9. Remove extra spaces
    text = ' '.join(text.split())
    
    return text


def loose_clean(text):
    """
    Loose cleaning for PhoBERT (minimal preprocessing)
    - Decode emojis to text
    - Word segmentation (but split underscores for BERT)
    - KEEP ORIGINAL CASE
    - Remove URLs, emails (replace with placeholders)
    - Remove HTML tags
    - Convert underscores to spaces (split compound words for BERT tokenizer)
    - KEEP NUMBERS
    - KEEP special characters, hashtags
    - Remove extra spaces only
    """
    if pd.isna(text) or text == '':
        return ''
    
    # 1. Decode emojis
    text = emoji.replace_emoji(text, replace="")
    
    # 2. Vietnamese word segmentation
    text = ViTokenizer.tokenize(text)
    
    # 3. Remove URLs (replace with placeholder)
    text = re.sub(r'http\S+|www\S+', '[URL]', text)
    text = re.sub(r'<\s*url\s*>', '[URL]', text)
    
    # 4. Remove emails (replace with placeholder)
    text = re.sub(r'\S+@\S+', '[EMAIL]', text)
    
    # 5. Remove HTML tags but keep content
    text = re.sub(r'<[^>]+>', '', text)
    
    # 6. Replace underscores with spaces (split compound words for BERT tokenizer)
    # Việt_Nam → Việt Nam (help BERT understand as two tokens)
    text = text.replace('_', ' ')
    
    # 7. Keep numbers, special characters, hashtags
    # Just remove excessive spaces
    text = ' '.join(text.split())
    
    return text

print(f"✅ Text cleaning functions UPDATED")
print(f"   strict: emoji_decode + word_segment(with_underscores) + lowercase + remove_numbers + remove_special_chars")
print(f"   loose:  emoji_decode + word_segment + keep_case + split_underscores + keep_numbers + keep_special_chars")

✅ Text cleaning functions UPDATED
   strict: emoji_decode + word_segment(with_underscores) + lowercase + remove_numbers + remove_special_chars
   loose:  emoji_decode + word_segment + keep_case + split_underscores + keep_numbers + keep_special_chars


## 3. Load Original Training Data (for TF-IDF fitting)

In [6]:
train_path = '../../data/splited/train_set.csv'

df_train = pd.read_csv(train_path)
y_train = df_train['label'].values

print(f"✅ Original TRAIN set loaded: {df_train.shape}")
print(f"   Labels: {(y_train==0).sum()} fake / {(y_train==1).sum()} real")
print(f"   Purpose: Fit TF-IDF & SVD (same as model training)")
text_cols = [col for col in df_train.columns if 'text' in col.lower()]
print(f"   Text columns available: {text_cols}")

✅ Original TRAIN set loaded: (3788, 28)
   Labels: 3143 fake / 645 real
   Purpose: Fit TF-IDF & SVD (same as model training)
   Text columns available: ['text_strict', 'text_loose']


In [7]:
# Load External Datasets

print("="*80)
print("LOADING EXTERNAL DATASETS")
print("="*80)

# 1. Load huynhtuan0106 (Train + Test combined)
ht_train = pd.read_csv('../../data/external/huynhtuan0106/train_data.csv')
ht_test = pd.read_csv('../../data/external/huynhtuan0106/test_data.csv')
df_huynhtuan = pd.concat([ht_train, ht_test], ignore_index=True)
df_huynhtuan.rename(columns={'content': 'text'}, inplace=True)

print(f"\n✅ huynhtuan0106 loaded")
print(f"   Train: {ht_train.shape}, Test: {ht_test.shape}")
print(f"   Combined: {df_huynhtuan.shape}")
print(f"   Labels: {(df_huynhtuan['label']==0).sum()} fake / {(df_huynhtuan['label']==1).sum()} real")

# 2. Load VNFD (Two files combined)
vnfd_223 = pd.read_csv('../../data/external/vnfd/vn_news_223_tdlfr.csv')
vnfd_226 = pd.read_csv('../../data/external/vnfd/vn_news_226_tlfr.csv')
df_vnfd = pd.concat([vnfd_223, vnfd_226], ignore_index=True)
df_vnfd = df_vnfd[['text', 'label']]  # Keep only text and label columns

print(f"\n✅ VNFD loaded")
print(f"   File 1 (223): {vnfd_223.shape}, File 2 (226): {vnfd_226.shape}")
print(f"   Combined: {df_vnfd.shape}")
print(f"   Labels: {(df_vnfd['label']==0).sum()} fake / {(df_vnfd['label']==1).sum()} real")

print(f"\n✅ All external datasets ready for processing!")


LOADING EXTERNAL DATASETS

✅ huynhtuan0106 loaded
   Train: (1124, 2), Test: (282, 2)
   Combined: (1406, 2)
   Labels: 703 fake / 703 real

✅ VNFD loaded
   File 1 (223): (223, 3), File 2 (226): (226, 2)
   Combined: (449, 2)
   Labels: 246 fake / 203 real

✅ All external datasets ready for processing!


## 4. Fit TF-IDF & SVD on Original Training Data

In [8]:
texts_train_strict = df_train['text_strict'].fillna('').tolist()

print(f"✅ Extracted {len(texts_train_strict)} training texts for TF-IDF fitting")

# Fit TF-IDF on ORIGINAL training data (same as model training)
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=2, sublinear_tf=True)
X_train_tfidf_full = tfidf.fit_transform(texts_train_strict)

# Fit SVD on ORIGINAL training data
svd = TruncatedSVD(n_components=900, random_state=42)
X_train_tfidf = svd.fit_transform(X_train_tfidf_full)

print(f"✅ TF-IDF & SVD fitted on original training data")
print(f"   TF-IDF vocabulary size: {len(tfidf.vocabulary_)}")
print(f"   SVD n_components: {svd.n_components}")
print(f"   SVD explained variance ratio: {svd.explained_variance_ratio_.sum():.4f}")

✅ Extracted 3788 training texts for TF-IDF fitting
✅ TF-IDF & SVD fitted on original training data
   TF-IDF vocabulary size: 57103
   SVD n_components: 900
   SVD explained variance ratio: 0.4905


## 5. Load External Datasets

In [9]:
# Apply cleaning to both external datasets

print("="*80)
print("APPLYING STRICT & LOOSE CLEANING")
print("="*80)

# huynhtuan0106
print("\n1️⃣  huynhtuan0106 - Applying text cleaning...")
df_huynhtuan['text_strict'] = df_huynhtuan['text'].apply(strict_clean)
df_huynhtuan['text_loose'] = df_huynhtuan['text'].apply(loose_clean)

# Extract texts
ht_texts_strict = df_huynhtuan['text_strict'].fillna('').tolist()
ht_texts_loose = df_huynhtuan['text_loose'].fillna('').tolist()
ht_labels = df_huynhtuan['label'].values

# Transform strict text with TF-IDF
ht_tfidf_full = tfidf.transform(ht_texts_strict)
ht_tfidf = svd.transform(ht_tfidf_full)

print(f"   ✅ Text cleaned: {df_huynhtuan.shape[0]} samples")
print(f"   ✅ TF-IDF transformed: {ht_tfidf.shape}")
print(f"   Example:")
print(f"      Original: {df_huynhtuan['text'].iloc[0][:60]}...")
print(f"      Strict:   {df_huynhtuan['text_strict'].iloc[0][:60]}...")
print(f"      Loose:    {df_huynhtuan['text_loose'].iloc[0][:60]}...")

# VNFD
print("\n2️⃣  VNFD - Applying text cleaning...")
df_vnfd['text_strict'] = df_vnfd['text'].apply(strict_clean)
df_vnfd['text_loose'] = df_vnfd['text'].apply(loose_clean)

# Extract texts
vnfd_texts_strict = df_vnfd['text_strict'].fillna('').tolist()
vnfd_texts_loose = df_vnfd['text_loose'].fillna('').tolist()
vnfd_labels = df_vnfd['label'].values

# Transform strict text with TF-IDF
vnfd_tfidf_full = tfidf.transform(vnfd_texts_strict)
vnfd_tfidf = svd.transform(vnfd_tfidf_full)

print(f"   ✅ Text cleaned: {df_vnfd.shape[0]} samples")
print(f"   ✅ TF-IDF transformed: {vnfd_tfidf.shape}")
print(f"   Example:")
print(f"      Original: {df_vnfd['text'].iloc[0][:60]}...")
print(f"      Strict:   {df_vnfd['text_strict'].iloc[0][:60]}...")
print(f"      Loose:    {df_vnfd['text_loose'].iloc[0][:60]}...")

print(f"\n✅ Text cleaning completed for both datasets!")

APPLYING STRICT & LOOSE CLEANING

1️⃣  huynhtuan0106 - Applying text cleaning...
   ✅ Text cleaned: 1406 samples
   ✅ TF-IDF transformed: (1406, 900)
   Example:
      Original: u là chời khách du lịch bay đêm đến sài gòn được giảm giá ho...
      Strict:   u là chời khách du_lịch bay đêm đến sài_gòn được giảm_giá ho...
      Loose:    u là chời khách du lịch bay đêm đến sài gòn được giảm giá ho...

2️⃣  VNFD - Applying text cleaning...
   ✅ Text cleaned: 449 samples
   ✅ TF-IDF transformed: (449, 900)
   Example:
      Original: Thủ tướng Abe cúi đầu xin lỗi vì hành động phi thể thao của ...
      Strict:   thủ_tướng abe cúi đầu xin_lỗi vì hành_động phi thể_thao của ...
      Loose:    Thủ tướng Abe cúi đầu xin lỗi vì hành động phi thể thao của ...

✅ Text cleaning completed for both datasets!


## 8. Extract PhoBERT Embeddings

In [10]:
# Extract PhoBERT embeddings from loose_clean text

print("="*80)
print("EXTRACTING PhoBERT EMBEDDINGS")
print("="*80)

# Load PhoBERT model
model_name = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
phobert = AutoModel.from_pretrained(model_name).to(device)
phobert.eval()

print(f"\n✅ PhoBERT model loaded: {model_name}")

def extract_phobert_embedding(texts, batch_size=8):
    """Extract [CLS] token embedding from PhoBERT"""
    embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting embeddings"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = phobert(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # [CLS] token (first token)
            embeddings.append(cls_embeddings.cpu().numpy())
    
    return np.vstack(embeddings)

# Extract for huynhtuan0106
print(f"\n1️⃣  Extracting PhoBERT for huynhtuan0106 ({len(ht_texts_loose)} samples)...")
ht_phobert = extract_phobert_embedding(ht_texts_loose, batch_size=8)
print(f"   ✅ Shape: {ht_phobert.shape}")

# Extract for VNFD
print(f"\n2️⃣  Extracting PhoBERT for VNFD ({len(vnfd_texts_loose)} samples)...")
vnfd_phobert = extract_phobert_embedding(vnfd_texts_loose, batch_size=8)
print(f"   ✅ Shape: {vnfd_phobert.shape}")

print(f"\n✅ PhoBERT extraction completed!")

# Clean up GPU memory
del phobert
del tokenizer
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print(f"   ✅ GPU memory cleaned")

EXTRACTING PhoBERT EMBEDDINGS


Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



✅ PhoBERT model loaded: vinai/phobert-base-v2

1️⃣  Extracting PhoBERT for huynhtuan0106 (1406 samples)...


Extracting embeddings: 100%|██████████| 176/176 [00:26<00:00,  6.77it/s]


   ✅ Shape: (1406, 768)

2️⃣  Extracting PhoBERT for VNFD (449 samples)...


Extracting embeddings: 100%|██████████| 57/57 [00:12<00:00,  4.51it/s]


   ✅ Shape: (449, 768)

✅ PhoBERT extraction completed!
   ✅ GPU memory cleaned


## 9. Combine Embeddings

In [11]:
print("="*80)
print("COMBINING TF-IDF + PhoBERT EMBEDDINGS")
print("="*80)

# Combine TF-IDF (900D) + PhoBERT (768D) for both datasets
ht_combined = np.hstack([ht_tfidf, ht_phobert])
vnfd_combined = np.hstack([vnfd_tfidf, vnfd_phobert])

print(f"\n✅ Combined embeddings created (TF-IDF 900D + PhoBERT 768D = 1668D)")
print(f"   huynhtuan0106: {ht_combined.shape}")
print(f"   VNFD: {vnfd_combined.shape}")

# Verify
assert ht_combined.shape[1] == 1668, f"Expected 1668D, got {ht_combined.shape[1]}"
assert vnfd_combined.shape[1] == 1668, f"Expected 1668D, got {vnfd_combined.shape[1]}"
print(f"\n✅ Dimensions verified: 1668D correct")

COMBINING TF-IDF + PhoBERT EMBEDDINGS

✅ Combined embeddings created (TF-IDF 900D + PhoBERT 768D = 1668D)
   huynhtuan0106: (1406, 1668)
   VNFD: (449, 1668)

✅ Dimensions verified: 1668D correct


## 10. Load Model & Scaler

In [12]:
print("\n" + "="*80)
print("LOADING MODEL & SCALER")
print("="*80)

model_dir = os.path.abspath('../../model/machine_learned')
print(f"\n📂 Model directory: {model_dir}")

if os.path.exists(model_dir):
    files = os.listdir(model_dir)
    print(f"\n📁 Files found: {len(files)}")
    for f in sorted(files):
        print(f"   - {f}")
else:
    raise FileNotFoundError(f"Model directory not found: {model_dir}")

# Load model and scaler
model_files = [f for f in files if f.startswith('Model_') and f.endswith('.pkl')]
scaler_files = [f for f in files if f.startswith('scaler_') and f.endswith('.pkl')]

latest_model_file = sorted(model_files)[-1]
latest_scaler_file = sorted(scaler_files)[-1]

best_model = joblib.load(os.path.join(model_dir, latest_model_file))
scaler = joblib.load(os.path.join(model_dir, latest_scaler_file))

print(f"\n✅ Model loaded: {latest_model_file}")
print(f"✅ Scaler loaded: {latest_scaler_file}")


LOADING MODEL & SCALER

📂 Model directory: d:\Vietnamese-Fake-News-Detection\model\machine_learned

📁 Files found: 3
   - Model_4_4_20260406_192428.pkl
   - Model_4_4_20260406_192428_metadata.txt
   - scaler_20260406_192428.pkl

✅ Model loaded: Model_4_4_20260406_192428.pkl
✅ Scaler loaded: scaler_20260406_192428.pkl


## 11. Predict on HuynhTuan0106

In [13]:
print("\n" + "="*80)
print("EVALUATION ON huynhtuan0106 (Threshold = 0.45)")
print("="*80)

# Scale embeddings
ht_combined_scaled = scaler.transform(ht_combined)

# Make predictions
ht_proba = best_model.predict_proba(ht_combined_scaled)[:, 1]

THRESHOLD = 0.45
ht_pred = (ht_proba >= THRESHOLD).astype(int)

# Evaluate
f1_ht = f1_score(ht_labels, ht_pred, average='weighted')
auc_ht = roc_auc_score(ht_labels, ht_proba)
acc_ht = accuracy_score(ht_labels, ht_pred)
prec_ht = precision_score(ht_labels, ht_pred, average='weighted')
rec_ht = recall_score(ht_labels, ht_pred, average='weighted')
cm_ht = confusion_matrix(ht_labels, ht_pred)

print(f"\n✅ Predictions made: {len(ht_pred)} samples")
print(f"   Fake (0): {(ht_pred==0).sum()} | Real (1): {(ht_pred==1).sum()}")

print(f"\n📊 Performance Metrics:")
print(f"   F1 Score:    {f1_ht:.4f}")
print(f"   AUC-ROC:     {auc_ht:.4f}")
print(f"   Accuracy:    {acc_ht:.4f}")
print(f"   Precision:   {prec_ht:.4f}")
print(f"   Recall:      {rec_ht:.4f}")

print(f"\n📊 Confusion Matrix:")
print(f"   TN: {cm_ht[0,0]:3d} | FP: {cm_ht[0,1]:3d}")
print(f"   FN: {cm_ht[1,0]:3d} | TP: {cm_ht[1,1]:3d}")
print(f"   Total Errors: {cm_ht[0,1] + cm_ht[1,0]:3d}")


EVALUATION ON huynhtuan0106 (Threshold = 0.45)

✅ Predictions made: 1406 samples
   Fake (0): 1029 | Real (1): 377

📊 Performance Metrics:
   F1 Score:    0.5806
   AUC-ROC:     0.6159
   Accuracy:    0.6031
   Precision:   0.6314
   Recall:      0.6031

📊 Confusion Matrix:
   TN: 587 | FP: 116
   FN: 442 | TP: 261
   Total Errors: 558


## 12. Predict on VNFD

In [14]:
print("\n" + "="*80)
print("EVALUATION ON VNFD (Threshold = 0.45)")
print("="*80)

# Scale embeddings
vnfd_combined_scaled = scaler.transform(vnfd_combined)

# Make predictions
vnfd_proba = best_model.predict_proba(vnfd_combined_scaled)[:, 1]

vnfd_pred = (vnfd_proba >= THRESHOLD).astype(int)

# Evaluate
f1_vnfd = f1_score(vnfd_labels, vnfd_pred, average='weighted')
auc_vnfd = roc_auc_score(vnfd_labels, vnfd_proba)
acc_vnfd = accuracy_score(vnfd_labels, vnfd_pred)
prec_vnfd = precision_score(vnfd_labels, vnfd_pred, average='weighted')
rec_vnfd = recall_score(vnfd_labels, vnfd_pred, average='weighted')
cm_vnfd = confusion_matrix(vnfd_labels, vnfd_pred)

print(f"\n✅ Predictions made: {len(vnfd_pred)} samples")
print(f"   Fake (0): {(vnfd_pred==0).sum()} | Real (1): {(vnfd_pred==1).sum()}")

print(f"\n📊 Performance Metrics:")
print(f"   F1 Score:    {f1_vnfd:.4f}")
print(f"   AUC-ROC:     {auc_vnfd:.4f}")
print(f"   Accuracy:    {acc_vnfd:.4f}")
print(f"   Precision:   {prec_vnfd:.4f}")
print(f"   Recall:      {rec_vnfd:.4f}")

print(f"\n📊 Confusion Matrix:")
print(f"   TN: {cm_vnfd[0,0]:3d} | FP: {cm_vnfd[0,1]:3d}")
print(f"   FN: {cm_vnfd[1,0]:3d} | TP: {cm_vnfd[1,1]:3d}")
print(f"   Total Errors: {cm_vnfd[0,1] + cm_vnfd[1,0]:3d}")


EVALUATION ON VNFD (Threshold = 0.45)

✅ Predictions made: 449 samples
   Fake (0): 444 | Real (1): 5

📊 Performance Metrics:
   F1 Score:    0.4124
   AUC-ROC:     0.5983
   Accuracy:    0.5590
   Precision:   0.7557
   Recall:      0.5590

📊 Confusion Matrix:
   TN: 246 | FP:   0
   FN: 198 | TP:   5
   Total Errors: 198


## 13. Final Comparison

In [15]:
print("\n" + "="*80)
print("EXTERNAL DATASET EVALUATION SUMMARY")
print("="*80)

comparison_df = pd.DataFrame({
    'Metric': ['F1 Score', 'AUC-ROC', 'Accuracy', 'Precision', 'Recall', 'Total Errors', 'FP', 'FN'],
    'huynhtuan0106': [
        f'{f1_ht:.4f}', f'{auc_ht:.4f}', f'{acc_ht:.4f}', f'{prec_ht:.4f}', f'{rec_ht:.4f}',
        f'{cm_ht[0,1] + cm_ht[1,0]}', f'{cm_ht[0,1]}', f'{cm_ht[1,0]}'
    ],
    'VNFD': [
        f'{f1_vnfd:.4f}', f'{auc_vnfd:.4f}', f'{acc_vnfd:.4f}', f'{prec_vnfd:.4f}', f'{rec_vnfd:.4f}',
        f'{cm_vnfd[0,1] + cm_vnfd[1,0]}', f'{cm_vnfd[0,1]}', f'{cm_vnfd[1,0]}'
    ]
})

print("\n📊 Comparison Table:")
print(comparison_df.to_string(index=False))

print(f"\n\n" + "="*80)
print("GENERALIZATION ASSESSMENT")
print("="*80)

print(f"\n✅ Model: Model_4_4 (256→128→64 MLP)")
print(f"   Threshold: 0.45")
print(f"   Test Protocol: External datasets (generalization test)")

print(f"\n📈 External Dataset Performance:")
print(f"   huynhtuan0106: Accuracy {acc_ht:.4f}, F1 {f1_ht:.4f}")
print(f"   VNFD:          Accuracy {acc_vnfd:.4f}, F1 {f1_vnfd:.4f}")

avg_acc = (acc_ht + acc_vnfd) / 2
print(f"\n   Average Accuracy: {avg_acc:.4f}")

print(f"\n💡 Interpretation:")
if avg_acc >= 0.85:
    print(f"   ✅ STRONG GENERALIZATION - Model performs well on external data")
elif avg_acc >= 0.80:
    print(f"   ✅ GOOD GENERALIZATION - Model generalizes reasonably well")
else:
    print(f"   ⚠️  LIMITED GENERALIZATION - Model struggles on external data")
    print(f"   ⚠️  May need retraining on more diverse datasets")

print(f"\n✅ External evaluation complete!")


EXTERNAL DATASET EVALUATION SUMMARY

📊 Comparison Table:
      Metric huynhtuan0106   VNFD
    F1 Score        0.5806 0.4124
     AUC-ROC        0.6159 0.5983
    Accuracy        0.6031 0.5590
   Precision        0.6314 0.7557
      Recall        0.6031 0.5590
Total Errors           558    198
          FP           116      0
          FN           442    198


GENERALIZATION ASSESSMENT

✅ Model: Model_4_4 (256→128→64 MLP)
   Threshold: 0.45
   Test Protocol: External datasets (generalization test)

📈 External Dataset Performance:
   huynhtuan0106: Accuracy 0.6031, F1 0.5806
   VNFD:          Accuracy 0.5590, F1 0.4124

   Average Accuracy: 0.5811

💡 Interpretation:
   ⚠️  LIMITED GENERALIZATION - Model struggles on external data
   ⚠️  May need retraining on more diverse datasets

✅ External evaluation complete!
